# PixelWorld-0.5.1 — deterministische Weltübergänge

0.5.1 entfernt den nicht lernbaren Seed-Token-Kopf. Folgewelten werden deterministisch aus Welt-Seed, Slot-ID, Trigger-Typ und Story-State abgeleitet.

Geometrie, Presence, Klasse, Aktion und Trigger bleiben gegenüber 0.5 unverändert, damit der Architekturfix isoliert messbar bleibt.

In [ ]:
# Notebook-Abhängigkeiten installieren.
%pip install -q ipympl
# Google Colab blockiert Drittanbieter-Widgets standardmäßig.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print('Colab-Widget-Unterstützung aktiviert.')
except ImportError:
    pass
# Falls PyTorch/NumPy/Matplotlib fehlen:
# %pip install torch numpy matplotlib pillow

import hashlib, random
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. Gemeinsame Repräsentation

Jede Welt besitzt dieselbe räumliche Auflösung. Die Objekt-ID verbindet sichtbare Pixel mit maschinenlesbarer Logik. Trigger werden objektweise definiert und nicht redundant pro Pixel gespeichert.

In [ ]:
SIZE = 64
MAX_SLOTS = 8
SLOT_LATENT_DIM = 6
LAYOUT_DIM = 4 + MAX_SLOTS*SLOT_LATENT_DIM
THEMES = ['pirate', 'scifi', 'horror', 'fantasy', 'noir']
CLASSES = {'void': 0, 'wall': 1, 'floor': 2, 'door': 3, 'npc': 4, 'object': 5, 'portal': 6}
OBJECT_CLASSES = ['door', 'npc', 'object', 'portal']
ACTIONS = ['LOOK', 'USE', 'SCAN']
TRIGGER_TYPES = ['NONE', 'WORLD', 'STORY', 'SECRET']
CLASS_SIZES = {'door': (7,16), 'npc': (5,9), 'object': (4,5), 'portal': (6,8)}
PALETTES = {
    'pirate':  [(12,10,18),(70,42,28),(122,78,43),(145,42,35),(218,166,91),(37,115,88),(92,45,125)],
    'scifi':   [(5,8,18),(24,45,66),(58,80,92),(230,80,70),(100,235,220),(230,190,60),(110,75,220)],
    'horror':  [(7,5,8),(38,27,36),(72,54,60),(130,25,35),(190,155,135),(65,100,58),(100,25,105)],
    'fantasy': [(12,16,22),(58,70,52),(108,99,60),(155,75,45),(235,190,95),(50,130,75),(125,70,180)],
    'noir':    [(5,5,7),(35,37,42),(85,88,92),(180,45,40),(205,205,185),(60,110,115),(105,65,120)]
}

@dataclass
class World:
    prompt: str
    theme: str
    rgb: np.ndarray
    semantic: np.ndarray
    object_map: np.ndarray
    walkable: np.ndarray
    interaction: np.ndarray
    room_bbox: tuple
    objects: dict

def world_seed(text):
    return int(hashlib.sha256(text.encode()).hexdigest()[:8], 16)

def layout_from_seed(seed):
    # Derselbe Seed liefert im Generator und im Modell denselben latenten Layout-Vektor.
    return np.random.default_rng(seed).random(LAYOUT_DIM).astype(np.float32)

def transition_seed(current_seed, slot, trigger_type, story_state=0):
    # Identität wird deterministisch abgeleitet und nicht vom Modell erraten.
    return world_seed(f'{current_seed}:{slot}:{trigger_type}:{story_state}')

def generate_world(prompt, seed=None):
    seed = world_seed(prompt) if seed is None else seed
    rng = np.random.default_rng(seed)
    layout = layout_from_seed(seed)
    theme = next((t for t in THEMES if t in prompt.lower()), THEMES[seed % len(THEMES)])
    sem = np.full((SIZE, SIZE), CLASSES['wall'], np.int64)
    x0, y0 = 4 + int(layout[0]*6), 6 + int(layout[1]*7)
    x1, y1 = 55 + int(layout[2]*5), 53 + int(layout[3]*6)
    sem[y0:y1, x0:x1] = CLASSES['floor']
    obj = np.zeros((SIZE, SIZE), np.int64)
    objects = {}

    def place(oid, kind, x, y, w, h, action, label):
        sem[y:y+h, x:x+w] = CLASSES[kind]
        obj[y:y+h, x:x+w] = oid
        trigger = world_seed(f'{seed}:{oid}:{label}')
        objects[oid] = {'label': label, 'class': kind, 'action': action, 'next_seed': trigger,
                        'bbox': [x, y, w, h]}

    anchors = [(3,3), (14,3), (25,3), (3,16), (14,16), (25,16), (3,29)]
    expected_pixels = {}
    for slot in range(MAX_SLOTS):
        oid = slot + 1
        values = layout[4 + slot*SLOT_LATENT_DIM : 4 + (slot+1)*SLOT_LATENT_DIM]
        is_present = slot == 0 or values[0] > 0.30
        if not is_present: continue
        class_name = 'door' if slot == 0 else OBJECT_CLASSES[1 + min(2, int(values[1]*3))]
        w,h = CLASS_SIZES[class_name]
        if slot == 0:
            x = x1 - w
            y = y0 + 2 + int(values[3] * max(1, y1-y0-h-3))
        else:
            ax,ay = anchors[slot-1]
            x = x0 + ax + int(values[2]*3)
            y = y0 + ay + int(values[3]*2)
        action_id = min(len(ACTIONS)-1, int(values[4]*len(ACTIONS)))
        trigger_id = min(len(TRIGGER_TYPES)-1, int(values[5]*len(TRIGGER_TYPES)))
        label = f'{class_name}_{oid}'
        place(oid, class_name, x, y, w, h, ACTIONS[action_id], label)
        objects[oid]['trigger_type'] = TRIGGER_TYPES[trigger_id]
        objects[oid]['next_seed'] = transition_seed(seed, slot, TRIGGER_TYPES[trigger_id])
        expected_pixels[oid] = w*h
    actual_pixels = {oid: int((obj == oid).sum()) for oid in expected_pixels}
    if actual_pixels != expected_pixels:
        raise RuntimeError(f'Ungültige überlappende Objekte: {actual_pixels}')
    walk = (sem == CLASSES['floor']).astype(np.uint8)
    interaction = (obj > 0).astype(np.uint8)
    palette = np.asarray(PALETTES[theme], dtype=np.uint8)
    rgb = palette[sem]
    noise = rng.integers(-7, 8, rgb.shape, dtype=np.int16)
    rgb = np.clip(rgb.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return World(prompt, theme, rgb, sem, obj, walk, interaction, (x0,y0,x1,y1), objects)

world = generate_world('dark pirate tavern with a mysterious object')
world.objects

In [ ]:
def show_world(w):
    fig, ax = plt.subplots(1, 4, figsize=(15, 4))
    ax[0].imshow(w.rgb); ax[0].set_title(f'RGB — {w.theme}')
    ax[1].imshow(w.semantic, cmap='tab10', vmin=0, vmax=9); ax[1].set_title('Semantic Map')
    ax[2].imshow(w.object_map, cmap='nipy_spectral'); ax[2].set_title('Object Map')
    ax[3].imshow(w.interaction, cmap=ListedColormap(['black', 'gold'])); ax[3].set_title('Interaction Map')
    for a in ax: a.axis('off')
    plt.tight_layout()

show_world(world)

## 2. Scene-Graph-Datensatz und Object-Slot-Modell

0.5.1 verwendet acht kanonische Slot Queries. Jeder vorhandene Slot trägt Klasse, relative Position, Aktion und Trigger-Typ; nicht vorhandene Slots werden maskiert. Die Folgewelt-ID wird außerhalb des Modells deterministisch abgeleitet.

In [ ]:
def prompt_vector(prompt):
    p = prompt.lower()
    theme = next((t for t in THEMES if t in p), THEMES[world_seed(p) % len(THEMES)])
    v = np.zeros(len(THEMES) + 8, np.float32)
    v[THEMES.index(theme)] = 1
    for i, word in enumerate(['dark','bright','door','npc','object','mysterious','old','red']):
        v[len(THEMES)+i] = float(word in p)
    return v

def condition_vector(prompt, seed):
    return np.concatenate([prompt_vector(prompt), layout_from_seed(seed)]).astype(np.float32)

SLOT_IDS = list(range(1, MAX_SLOTS+1))

def scene_targets(world):
    room = np.asarray(world.room_bbox, dtype=np.int64)
    positions = np.zeros((MAX_SLOTS, 2), dtype=np.int64)
    presence = np.zeros(MAX_SLOTS, dtype=np.float32)
    class_ids = np.zeros(MAX_SLOTS, dtype=np.int64)
    action_ids = np.zeros(MAX_SLOTS, dtype=np.int64)
    trigger_ids = np.zeros(MAX_SLOTS, dtype=np.int64)
    for slot, oid in enumerate(SLOT_IDS):
        if oid in world.objects:
            x,y,_,_ = world.objects[oid]['bbox']
            positions[slot] = [x-world.room_bbox[0], y-world.room_bbox[1]]
            presence[slot] = 1.0
            meta = world.objects[oid]
            class_ids[slot] = OBJECT_CLASSES.index(meta['class'])
            action_ids[slot] = ACTIONS.index(meta['action'])
            trigger_ids[slot] = TRIGGER_TYPES.index(meta['trigger_type'])
    return room, positions, presence, class_ids, action_ids, trigger_ids

class PixelWorldDataset(Dataset):
    def __init__(self, n=10000):
        self.prompts = []
        adjectives = ['dark', 'bright', 'old', 'mysterious']
        for i in range(n):
            self.prompts.append(f'{adjectives[i % 4]} {THEMES[i % 5]} room with red door npc object {i}')
    def __len__(self): return len(self.prompts)
    def __getitem__(self, i):
        p = self.prompts[i]; seed = i + 1000; w = generate_world(p, seed=seed)
        room, positions, presence, classes, actions, triggers = scene_targets(w)
        return (torch.tensor(condition_vector(p, seed)), torch.tensor(room),
                torch.tensor(positions), torch.tensor(presence), torch.tensor(classes),
                torch.tensor(actions), torch.tensor(triggers))

COORD_CLASSES = SIZE + 1

class SceneGraphNet(nn.Module):
    def __init__(self, condition_dim=len(THEMES)+8+LAYOUT_DIM, slots=MAX_SLOTS, hidden=320):
        super().__init__()
        self.slots = slots
        self.world_encoder = nn.Sequential(
            nn.Linear(condition_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU())
        self.room_head = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 4*COORD_CLASSES))
        self.slot_queries = nn.Embedding(slots, hidden)
        self.slot_decoder = nn.Sequential(
            nn.Linear(2*hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU())
        self.position_head = nn.Linear(hidden, 2*COORD_CLASSES)
        self.class_head = nn.Linear(hidden, len(OBJECT_CLASSES))
        self.action_head = nn.Linear(hidden, len(ACTIONS))
        self.trigger_head = nn.Linear(hidden, len(TRIGGER_TYPES))
        self.presence_encoder = nn.Sequential(
            nn.Linear(condition_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU())
        self.presence_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, slots))

    def forward(self, condition):
        world = self.world_encoder(condition)
        queries = self.slot_queries.weight[None].expand(condition.shape[0], -1, -1)
        context = world[:,None,:].expand(-1, self.slots, -1)
        slots = self.slot_decoder(torch.cat([context, queries], dim=-1))
        room_logits = self.room_head(world).reshape(-1, 4, COORD_CLASSES)
        position_logits = self.position_head(slots).reshape(-1, self.slots, 2, COORD_CLASSES)
        presence_logits = self.presence_head(self.presence_encoder(condition))
        return (room_logits, position_logits, presence_logits, self.class_head(slots),
                self.action_head(slots), self.trigger_head(slots))

model = SceneGraphNet().to(DEVICE)
sum(p.numel() for p in model.parameters())

In [ ]:
# 0.5.1: Multi-Task-Training des vollständigen interaktiven Scene Graphs.
loader = DataLoader(PixelWorldDataset(10000), batch_size=128, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
presence_loss = nn.BCEWithLogitsLoss(reduction='none')
category_loss = nn.CrossEntropyLoss(reduction='none')
coordinate_values = torch.arange(COORD_CLASSES, dtype=torch.float32, device=DEVICE)

def ordinal_coordinate_loss(logits, target, sigma=1.0):
    distances = coordinate_values - target[...,None].float()
    soft_target = torch.exp(-0.5 * (distances / sigma)**2)
    soft_target = soft_target / soft_target.sum(-1, keepdim=True)
    soft_ce = -(soft_target * logits.log_softmax(-1)).sum(-1)
    expected = (logits.softmax(-1) * coordinate_values).sum(-1)
    geometric = F.smooth_l1_loss(expected, target.float(), reduction='none') / SIZE
    return soft_ce + 4.0 * geometric

def masked_category_loss(logits, target, presence):
    errors = category_loss(logits.flatten(0,1), target.flatten()).reshape_as(target)
    return (errors * presence).sum() / presence.sum().clamp_min(1)

EPOCHS = 40

for epoch in range(EPOCHS):
    model.train(); total = 0.0
    parts = np.zeros(6, dtype=np.float64)
    for condition, room_t, positions_t, presence_t, classes_t, actions_t, triggers_t in loader:
        condition, room_t = condition.to(DEVICE), room_t.to(DEVICE)
        positions_t, presence_t = positions_t.to(DEVICE), presence_t.to(DEVICE)
        classes_t, actions_t = classes_t.to(DEVICE), actions_t.to(DEVICE)
        triggers_t = triggers_t.to(DEVICE)
        outputs = model(condition)
        room_logits, position_logits, presence_logits, class_logits, action_logits, trigger_logits = outputs
        room_l = ordinal_coordinate_loss(room_logits, room_t).mean()
        position_error = ordinal_coordinate_loss(position_logits, positions_t).mean(-1)
        position_l = (position_error * presence_t).sum() / presence_t.sum().clamp_min(1)
        p_weights = torch.where(presence_t > 0.5, 1.0, 2.0)
        presence_l = (presence_loss(presence_logits, presence_t) * p_weights).mean()
        class_l = masked_category_loss(class_logits, classes_t, presence_t)
        action_l = masked_category_loss(action_logits, actions_t, presence_t)
        trigger_l = masked_category_loss(trigger_logits, triggers_t, presence_t)
        loss = room_l + 4.0*position_l + presence_l + class_l + action_l + trigger_l
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item(); parts += [room_l.item(), position_l.item(), presence_l.item(),
                                      class_l.item(), action_l.item(), trigger_l.item()]
    parts /= len(loader)
    print(f'Epoch {epoch+1:02d}: loss={total/len(loader):.3f} room={parts[0]:.3f} pos={parts[1]:.3f} '
          f'presence={parts[2]:.3f} class={parts[3]:.3f} action={parts[4]:.3f} '
          f'trigger={parts[5]:.3f}')

In [ ]:
# Modellvorhersage visualisieren
test_prompt = 'dark noir room with red door npc mysterious object'
test_seed = 424242
truth = generate_world(test_prompt, test_seed)
def predict_scene(model, prompt, seed):
    condition = torch.tensor(condition_vector(prompt, seed))[None].to(DEVICE)
    model.eval()
    with torch.no_grad():
        outputs = model(condition)
        room_logits, position_logits, presence_logits, class_logits, action_logits, trigger_logits = outputs
    def decode(logits):
        expected = (logits.softmax(-1) * coordinate_values).sum(-1)
        return expected.round().clamp(0, SIZE).to(torch.int64)
    return (decode(room_logits[0]).cpu().numpy(), decode(position_logits[0]).cpu().numpy(),
            presence_logits[0].sigmoid().cpu().numpy(), class_logits[0].argmax(-1).cpu().numpy(),
            action_logits[0].argmax(-1).cpu().numpy(), trigger_logits[0].argmax(-1).cpu().numpy())

def absolute_boxes(room, positions, class_ids):
    top_left = np.asarray(room[:2], dtype=int)
    xy = np.asarray(positions, dtype=int) + top_left[None,:]
    sizes = np.asarray([CLASS_SIZES[OBJECT_CLASSES[int(cid)]] for cid in class_ids], dtype=int)
    return np.concatenate([xy, sizes], axis=1)

def rasterize_scene(room, positions, presence, class_ids):
    sem = np.full((SIZE,SIZE), CLASSES['wall'], dtype=np.int64)
    obj = np.zeros((SIZE,SIZE), dtype=np.int64)
    x0,y0,x1,y1 = np.asarray(room, dtype=int)
    x0,y0 = np.clip([x0,y0], 0, SIZE-2); x1,y1 = np.clip([x1,y1], [x0+1,y0+1], SIZE)
    sem[y0:y1,x0:x1] = CLASSES['floor']
    boxes = absolute_boxes(np.asarray([x0,y0,x1,y1]), positions, class_ids)
    for slot, oid in enumerate(SLOT_IDS):
        if presence[slot] < 0.5: continue
        class_name = OBJECT_CLASSES[int(class_ids[slot])]
        x,y,w,h = np.asarray(boxes[slot], dtype=int)
        x,y = np.clip([x,y], 0, SIZE-1); w,h = np.clip([w,h], 1, SIZE)
        x2,y2 = min(SIZE,x+w), min(SIZE,y+h)
        sem[y:y2,x:x2] = CLASSES[class_name]
        obj[y:y2,x:x2] = oid
    return sem, obj, (obj > 0).astype(np.uint8)

pred_room, pred_positions, pred_presence, pred_classes, pred_actions, pred_triggers = predict_scene(model, test_prompt, test_seed)
pred_sem, pred_obj, pred_int = rasterize_scene(pred_room, pred_positions, pred_presence, pred_classes)
fig, ax = plt.subplots(2,3,figsize=(12,8))
for a, data, title in zip(ax[0], [truth.semantic,truth.object_map,truth.interaction], ['Ziel: Semantik','Ziel: Objekte','Ziel: Interaktion']):
    a.imshow(data, cmap='tab10'); a.set_title(title); a.axis('off')
for a, data, title in zip(ax[1], [pred_sem,pred_obj,pred_int], ['Modell: Semantik','Modell: Objekte','Modell: Interaktion']):
    a.imshow(data, cmap='tab10'); a.set_title(title); a.axis('off')
plt.tight_layout(); plt.show()
print('Vorhergesagte Object Slots:')
pred_boxes = absolute_boxes(pred_room, pred_positions, pred_classes)
for slot in range(MAX_SLOTS):
    print(f"  slot_{slot}: presence={pred_presence[slot]:.3f} class={OBJECT_CLASSES[pred_classes[slot]]:6s} "
          f"action={ACTIONS[pred_actions[slot]]:4s} trigger={TRIGGER_TYPES[pred_triggers[slot]]:6s} "
          f"next_seed={transition_seed(test_seed, slot, TRIGGER_TYPES[pred_triggers[slot]])} "
          f"relative_xy={pred_positions[slot].tolist()} bbox={pred_boxes[slot].tolist()}")

def class_iou(pred, target, class_names):
    scores = {}
    for name, cid in class_names.items():
        p, t = pred == cid, target == cid
        union = np.logical_or(p,t).sum()
        scores[name] = float(np.logical_and(p,t).sum()/union) if union else float('nan')
    return scores

print('Semantic IoU:', {k: round(v,3) for k,v in class_iou(pred_sem, truth.semantic, CLASSES).items()})
print('Interaction IoU:', round(class_iou(pred_int, truth.interaction, {'interactive':1})['interactive'], 3))

# Robuste Bewertung über 20 im Training ungesehene Welten.
eval_seeds = [500_000 + i*7919 for i in range(20)]
iou_history = {name: [] for name in CLASSES if name != 'void'}
interaction_ious, interaction_precision, interaction_recall = [], [], []
room_mae, relative_position_mae, absolute_position_mae = [], [], []
presence_accuracy, exact_relative_positions, exact_absolute_positions = [], [], []
class_accuracy, action_accuracy, trigger_accuracy = [], [], []
for seed in eval_seeds:
    target = generate_world(test_prompt, seed)
    room_t, positions_t, presence_t, classes_t, actions_t, triggers_t = scene_targets(target)
    room_p, positions_p, presence_p, classes_p, actions_p, triggers_p = predict_scene(model, test_prompt, seed)
    p_sem, _, p_int = rasterize_scene(room_p, positions_p, presence_p, classes_p)
    scores = class_iou(p_sem, target.semantic, CLASSES)
    for name in iou_history:
        if not np.isnan(scores[name]): iou_history[name].append(scores[name])
    predicted, actual = p_int == 1, target.interaction == 1
    tp = np.logical_and(predicted, actual).sum()
    fp = np.logical_and(predicted, ~actual).sum()
    fn = np.logical_and(~predicted, actual).sum()
    interaction_ious.append(tp / max(1, tp+fp+fn))
    interaction_precision.append(tp / max(1, tp+fp))
    interaction_recall.append(tp / max(1, tp+fn))
    room_mae.append(np.abs(room_p-room_t).mean())
    present_mask = presence_t > 0.5
    absolute_t = positions_t + room_t[:2][None,:]
    absolute_p = positions_p + room_p[:2][None,:]
    relative_position_mae.append(np.abs(positions_p[present_mask]-positions_t[present_mask]).mean())
    absolute_position_mae.append(np.abs(absolute_p[present_mask]-absolute_t[present_mask]).mean())
    exact_relative_positions.append((positions_p[present_mask] == positions_t[present_mask]).mean())
    exact_absolute_positions.append((absolute_p[present_mask] == absolute_t[present_mask]).mean())
    presence_accuracy.append(((presence_p >= 0.5) == present_mask).mean())
    class_accuracy.append((classes_p[present_mask] == classes_t[present_mask]).mean())
    action_accuracy.append((actions_p[present_mask] == actions_t[present_mask]).mean())
    trigger_accuracy.append((triggers_p[present_mask] == triggers_t[present_mask]).mean())

print('Mean IoU (20 Welten):', {k: round(float(np.mean(v)),3) for k,v in iou_history.items()})
print('Interaktion — IoU:', round(float(np.mean(interaction_ious)),3),
      'Precision:', round(float(np.mean(interaction_precision)),3),
      'Recall:', round(float(np.mean(interaction_recall)),3))
print('Struktur — Raum-MAE:', round(float(np.mean(room_mae)),3), 'Pixel',
      'Presence Accuracy:', round(float(np.mean(presence_accuracy)),3))
print('Position — relativ MAE:', round(float(np.mean(relative_position_mae)),3), 'Pixel',
      'absolut MAE:', round(float(np.mean(absolute_position_mae)),3), 'Pixel')
print('Exakte X/Y-Koordinaten — relativ:', round(float(np.mean(exact_relative_positions)),3),
      'absolut:', round(float(np.mean(exact_absolute_positions)),3))
print('Slot Accuracy — Klasse:', round(float(np.mean(class_accuracy)),3),
      'Aktion:', round(float(np.mean(action_accuracy)),3),
      'Trigger:', round(float(np.mean(trigger_accuracy)),3))
# Determinismus-Test: gleiche Übergangsdaten müssen immer exakt denselben Seed ergeben.
transition_checks = [transition_seed(seed, slot, trigger, state)
                     for seed in eval_seeds for slot in range(MAX_SLOTS)
                     for trigger in TRIGGER_TYPES for state in (0, 1)]
transition_checks_2 = [transition_seed(seed, slot, trigger, state)
                       for seed in eval_seeds for slot in range(MAX_SLOTS)
                       for trigger in TRIGGER_TYPES for state in (0, 1)]
print('Deterministische Weltübergänge:', transition_checks == transition_checks_2,
      'geprüfte Kombinationen:', len(transition_checks))

## 3. Pixel anklicken → Folgewelt

Diese Zelle nutzt zunächst die Ground-Truth-Welt. Klick auf Tür, Figur, Objekt oder Portal. Der ausgewählte Pixel wird über die Object Map aufgelöst; `transition_seed` erzeugt aus Welt-Seed, Slot, Trigger und Story-State reproduzierbar die nächste Welt.

In [ ]:
# Interaktives Backend verwenden, wenn es verfügbar ist.
# Ohne ipympl bleibt scan_pixel(x, y) als zuverlässiger Fallback.
interactive_backend = False
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
    interactive_backend = True
except (ValueError, ImportError, ModuleNotFoundError):
    get_ipython().run_line_magic('matplotlib', 'inline')
    print('Klick-Backend nicht verfügbar. Nutze scan_pixel(x, y).')

current = generate_world('dark pirate room with mysterious object')
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(current.rgb, interpolation='nearest')
title = ax.set_title('Klicke einen interaktiven Pixel')
ax.axis('off')

def scan_pixel(x, y):
    global current
    x, y = int(x), int(y)
    if not (0 <= x < SIZE and 0 <= y < SIZE):
        raise ValueError(f'Koordinaten müssen zwischen 0 und {SIZE-1} liegen.')
    oid = int(current.object_map[y, x])
    if oid == 0:
        status = f'Pixel ({x},{y}): keine Interaktion'
    else:
        meta = current.objects[oid]
        next_prompt = f"{current.theme} world inside {meta['label']}"
        current = generate_world(next_prompt, meta['next_seed'])
        status = f"{meta['action']} {meta['label']} → neue Welt, Seed {meta['next_seed']}"
    print(status)
    if interactive_backend:
        im.set_data(current.rgb)
        title.set_text(status)
        fig.canvas.draw_idle()
    else:
        new_fig, new_ax = plt.subplots(figsize=(6,6))
        new_ax.imshow(current.rgb, interpolation='nearest')
        new_ax.set_title(status)
        new_ax.axis('off')
        plt.show()

def on_click(event):
    if event.xdata is not None and event.ydata is not None:
        scan_pixel(event.xdata, event.ydata)

def scan_object(object_id):
    ys, xs = np.where(current.object_map == object_id)
    if len(xs) == 0:
        print(f'Objekt {object_id} ist in dieser Welt nicht sichtbar.')
        return
    scan_pixel(int(np.median(xs)), int(np.median(ys)))

def list_interactions():
    for oid, meta in current.objects.items():
        ys, xs = np.where(current.object_map == oid)
        if len(xs):
            print(f"{oid}: {meta['label']} bei ungefähr ({int(np.median(xs))}, {int(np.median(ys))}) → {meta['action']}")

if interactive_backend:
    fig.canvas.mpl_connect('button_press_event', on_click)
plt.show()
list_interactions()
if not interactive_backend:
    print('Beispiel: scan_object(1) scannt die Tür unabhängig von ihrer Position.')

## Nächste Experimente

1. Deterministische Übergänge über mehrere Story-States und Pfade testen.
2. Kanonische Slot-Reihenfolge durch permutation-invariantes Matching ersetzen.
3. Pro Slot eine freie Pixelmaske und Sprite-ID erzeugen.
4. Beziehungen wie `on`, `inside`, `locked_by` und `leads_to` vorhersagen.
5. Den geplanten Editor als echte Datenquelle anbinden.
6. Titelbildschirme, Animation States und Pixel-Art-Renderer ergänzen.
7. Story-Constraints und dauerhaften World-State verbinden.